# 07 - J-lens experiments

**A playground, not a gate.** Nothing here is pre-registered, nothing here feeds
the P1-P5 grid, and no cell raises on a bad result. 01b is where the V1b gate
lives; this is where you poke the lens and find out what it does.

What it gives you:

- **Section 2** -- the infra. `readout`, `layer_table`, `track`, `probe`. Every
  helper is defined in this notebook, inline and hackable; nothing here imports
  a project module you would have to go and edit somewhere else.
- **Section 3** -- the Judy smoke test. Two forward passes that say whether the
  instrument works at all, so a strange result later is attributable.
- **Sections 4-6** -- 01b's **1-9 confidence** runs, copied over with the
  prompts printed. The 1-9 scale is used here on purpose: a rating is a
  **single token**, so the J-lens readout and the model's own distribution sit
  at the *identical* slot and can be compared directly. 01b now runs on 0-100,
  where the lens only ever sees the first digit -- fine for the gate, useless
  for playing.
- **Section 7** -- a scratch cell. Type a prompt, get the layer table.

> Everything below `probe` is a worked example of `probe`. If a section is in
> your way, delete it -- this notebook is not load-bearing.

In [1]:
# --- which model ----------------------------------------------------------
# Run this BEFORE the header cell; `get_model_config()` reads the env var at
# call time, so this is all it takes to switch scale.
#
# On this box (RTX 4090, 24 GB) `target` is the ceiling, not just the choice:
# gemma-3-12b-it is ~24 GB of bf16 weights against 24.5 GB of VRAM, so
# `escalate` does not fit here at any speed. 4b-it is ~8.6 GB, resident with
# room for activations, and is PLAN.md 5.0's experiment model regardless.
import gc, os

os.environ["NANDA_PRESET"] = "target"   # debug(270m) | main(1b) | target(4b) | escalate(12b)

# Drop a previously loaded model if this kernel already has one -- a second
# device_map="auto" load stacked on the first is how a 4090 OOMs at 4b.
for _name in ("model_jlens", "lens", "model"):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
    print("free VRAM:", round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GiB")
except (ImportError, RuntimeError):
    pass


free VRAM: 23.1 GiB


In [2]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

from nandaproj import config

cfg = config.get_model_config()      # NANDA_PRESET env var, defaults to debug
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())

preset: google/gemma-3-4b-it | 4B | bfloat16
device: cuda


In [3]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from nandaproj import synthetic, viz

tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    cfg.name,
    cache_dir=str(config.HF_CACHE),
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# gemma-3 4b and 12b are multimodal: they load as Gemma3ForConditionalGeneration
# and their config is a composite, so the language-model hyperparameters sit
# under `.text_config`. 270m and 1b are text-only Gemma3ForCausalLM and expose
# them at the top level. Resolve whichever this is -- reading the wrong one is
# either an AttributeError (loud) or the *vision* tower's depth (silent, and
# every `l >= N_LAYERS // 2` check downstream would then be quietly wrong).
TEXT_CFG = getattr(model.config, "text_config", model.config)
N_LAYERS = TEXT_CFG.num_hidden_layers
D_MODEL = TEXT_CFG.hidden_size

print(f"{type(model).__name__}: {N_LAYERS} layers, {D_MODEL} d_model")


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Gemma3ForConditionalGeneration: 34 layers, 2560 d_model


## 1. The lens

`lens.source_layers` is the ground truth for which layers have a fitted `J_l`,
and it is **not** always `range(n_layers)`: the 270m lens covers 0-16 of an
18-layer model, with nothing for the final layer. `layers=None` in `lens.apply`
resolves to exactly this list, so everything below sweeps `LAYERS` rather than
assuming the stack.

`n_prompts` is how many prompts the expectation `J_l = E[dh_final/dh_l]` was
averaged over (PLAN.md 4.1). It bounds how much *contextual* signal the lens can
retain, and it is the caveat to quote when a readout looks weaker than expected.

In [5]:
import jlens

model_jlens = jlens.from_hf(model, tok)
lens = jlens.JacobianLens.from_pretrained(
    config.LENS_REPO,
    filename=f"{cfg.lens_id}/jlens/Salesforce-wikitext/{cfg.lens_id}_jacobian_lens.pt",
)

LAYERS = list(lens.source_layers)
MISSING = sorted(set(range(N_LAYERS)) - set(LAYERS))
UPPER = [l for l in LAYERS if l >= N_LAYERS // 2]

print(f"model has {N_LAYERS} layers; lens fitted on {len(LAYERS)}: {LAYERS}")
print("no fitted Jacobian for:", MISSING or "none")
print(f"fitted from n_prompts={lens.n_prompts}, d_model={lens.d_model}")
print("upper half (what 'read late in the stack' means here):", UPPER)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model has 34 layers; lens fitted on 33: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
no fitted Jacobian for: [33]
fitted from n_prompts=546, d_model=2560
upper half (what 'read late in the stack' means here): [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


## 2. The infra

Four functions, in dependency order. `probe` is the one to reach for; the other
three are what it is made of and are worth calling directly when you want one
piece.

| | what it gives you |
|---|---|
| `readout(prompt)` | raw probability vectors per layer: J-lens, logit lens, model |
| `layer_table(prompt)` | the top-k at every layer, both lenses, side by side |
| `track(prompt, tokens)` | P(specific token) per layer, as a plot |
| `probe(prompt, tokens)` | both of the above in one call |

**Two seams to know about**, both inherited from `jlens` and both cheap to trip
over silently:

1. `lens.apply` takes a **single string** -- it does not batch. Every helper
   here loops. If a sweep feels slow, that is why.
2. `lens.apply` truncates at `max_seq_len=512` **without saying so**. Position
   `-1` would then be the 512th token rather than the end of your prompt, which
   is a null that looks like a finding. `readout` checks the length and raises.

In [6]:
MAX_SEQ_LEN = 512   # lens.apply's own default; readout enforces it out loud


def readout(prompt: str, layers=None, position: int = -1):
    """Probability vectors at one position, per layer, for both lenses.

    Returns `(j_probs, l_probs, final)`:
      j_probs  {layer: [vocab]}  -- J-lens, residual transported by J_l
      l_probs  {layer: [vocab]}  -- vanilla logit lens, identical activations
      final    [vocab]           -- the model's own next-token distribution

    The logit-lens pass is the control that matters: it says whether J-space is
    doing any work, or whether the token was legible from the raw residual
    anyway. Two forward passes per call, one per `use_jacobian` setting.
    """
    layers = list(layers) if layers is not None else LAYERS

    n_tokens = len(tok.encode(prompt, add_special_tokens=True))
    if n_tokens >= MAX_SEQ_LEN:
        raise ValueError(
            f"prompt is {n_tokens} tokens; lens.apply truncates at {MAX_SEQ_LEN} "
            f"and position {position} would no longer be the end of your prompt"
        )

    jl, ml, _ = lens.apply(model_jlens, prompt, layers=layers, positions=[position])
    ll, _, _ = lens.apply(model_jlens, prompt, layers=layers, positions=[position],
                          use_jacobian=False)
    soft = lambda t: torch.softmax(t.float(), dim=-1).cpu().numpy()
    return ({l: soft(jl[l][0]) for l in layers},
            {l: soft(ll[l][0]) for l in layers},
            soft(ml[0]))


def top_k(probs, k: int = 5):
    """`[(token_string, probability)]`, most likely first."""
    idx = np.argsort(probs)[-k:][::-1]
    return [(tok.decode([int(i)]), float(probs[i])) for i in idx]


def token_id(text: str) -> int:
    """The single token id for `text`, or a loud failure.

    Multi-token strings have no single slot to be read at, so every per-position
    metric here would be measuring something other than what you asked for.
    """
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(
            f"{text!r} is {len(ids)} tokens {[tok.decode([i]) for i in ids]}, not 1 -- "
            "pick one of those pieces, or use a scale where values are single tokens"
        )
    return int(ids[0])

In [7]:
def layer_table(prompt: str, k: int = 5, probs=None, show_p: bool = False):
    """Print the top-`k` at every fitted layer, J-lens against logit lens.

    The single most useful view in this notebook: it shows *where* in the stack
    a token appears and whether the Jacobian is what surfaced it.

    `probs` accepts a `readout(...)` triple so you can re-render a table without
    paying for the forward passes again.
    """
    j_probs, l_probs, final = probs if probs is not None else readout(prompt)

    def fmt(row):
        return ", ".join(f"{t!r}:{p:.2f}" for t, p in row) if show_p \
            else str([t for t, _ in row])

    print(f"{prompt!r}\n")
    print(f"{'layer':>5}  {'J-lens':<52}  logit lens")
    for l in LAYERS:
        print(f"{l:>5}  {fmt(top_k(j_probs[l], k)):<52}  {fmt(top_k(l_probs[l], k))}")
    print(f"\nmodel's own next token: {fmt(top_k(final, k))}")
    return j_probs, l_probs, final


def hits(prompt: str, target: str, k: int = 5, probs=None):
    """Layers where `target` is in the J-lens top-`k`. Whitespace-insensitive.

    Returned as the full list, not a first-hit index: presence is **not**
    monotone in layer, and reporting only the earliest hit hides a token that
    appears at L11 and is gone by L14.
    """
    j_probs, _, _ = probs if probs is not None else readout(prompt)
    want = target.strip()
    return [l for l in LAYERS
            if want in [t.strip() for t, _ in top_k(j_probs[l], k)]]

In [8]:
def track(prompt: str, tokens, probs=None, title: str = "", plot: bool = True):
    """P(each token) at every layer, J-lens vs logit lens.

    `layer_table` answers "is it in the top-5"; this answers "how much mass",
    which is the one that shows a token rising through the stack rather than
    appearing at a threshold.

    At most **two** tokens: `viz.SERIES` has four colour slots and each token
    costs two lines (one per lens).
    """
    if isinstance(tokens, str):
        tokens = [tokens]
    if len(tokens) > 2:
        raise ValueError(f"{len(tokens)} tokens x 2 lenses exceeds viz's 4 colour slots")

    j_probs, l_probs, final = probs if probs is not None else readout(prompt)

    series = {}
    for text in tokens:
        tid = token_id(text)
        series[f"J-lens {text!r}"] = [float(j_probs[l][tid]) for l in LAYERS]
        series[f"logit lens {text!r}"] = [float(l_probs[l][tid]) for l in LAYERS]

    if plot:
        viz.series_line(
            LAYERS, series, y_range=(0, 1),
            title=title or f"P(token) at the last position -- {prompt[:48]!r}",
            xaxis="layer", yaxis="probability",
        ).show()
    return series


def probe(prompt: str, tokens=None, k: int = 5, show_p: bool = False):
    """One call, one set of forward passes, everything: table then curves.

    The default entry point. `probe("...")` for a look; `probe("...", " Judy")`
    when you have a specific token in mind.
    """
    out = readout(prompt)
    layer_table(prompt, k=k, probs=out, show_p=show_p)
    if tokens:
        track(prompt, tokens, probs=out)
    return out

## 3. Smoke test - the Judy prompt

01's V1 prompt, verbatim. `" Judy"` is a name sitting in plain sight two clauses
back, so this is the easiest thing the lens will ever be asked to do: **copy**,
not compute.

Its only job is attribution. If Judy reads and a later section does not, the
instrument works and that section found something. If Judy does not read, the
lens, the wrapper, or `LAYERS` is broken and nothing below means anything.

On 270m-it, 01 saw `" Judy"` from L12 up.

In [9]:
JUDY = "Jake and Judy were talking to each other. Jake then handed his toy to"

judy_out = probe(JUDY, tokens=" Judy")

judy_layers = hits(JUDY, " Judy", probs=judy_out)
judy_upper = [l for l in judy_layers if l >= N_LAYERS // 2]
print(f"\n' Judy' in the J-lens top-5 at layers: {judy_layers or 'never'}")

# A warning, not a gate. This notebook does not raise -- 01b is where a failed
# check is supposed to stop the world.
if not judy_upper:
    print("!! the lens cannot read the easiest prompt there is. Check LAYERS, "
          "the jlens wrapper, and the lens file before trusting anything below.")

'Jake and Judy were talking to each other. Jake then handed his toy to'

layer  J-lens                                                logit lens
    0  ['\\"', '.\\"', '</strong>', ' \\"', '\\".']          ['ppling', 'ppled', 'othed', 'asting', 'ponym']
    1  ['<start_of_image>', '.\\"', '\\"', ' \\"', ' s']     ['ppling', 'ppled', 'othed', 'asting', 'ponym']
    2  ['.\\"', '<start_of_image>', '.}', '\\"', '.\\']      ['ppling', 'ppled', 'asting', 'othed', 'ponym']
    3  ['.\\"', '<start_of_image>', '\\"', '\\".', '</strong>']  ['ppling', 'ppled', 'asting', 'ponym', ' whom']
    4  ['<start_of_image>', '</strong>', '.\\"', '.}', '\\".']  ['َ', ' itse', ' respecto', ' whom', 'ppling']
    5  ['<start_of_image>', '�', '  ', '\\\\', '.\\\\']      ['َ', ' itse', ' respecto', ' लेकर', 'ppling']
    6  ['<start_of_image>', '�', ' \\"', ' the', '  ']       [' itse', 'َ', ' respecto', 'ppling', ' लेकर']
    7  ['<start_of_image>', ' \\"', ' $\\', ' the', " \\'"]  [' itse', ' respecto', 'ppl


' Judy' in the J-lens top-5 at layers: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


## 4. The 1-9 confidence scale

01b's confidence machinery, on the **1-9** scale. A 1-9 rating is a *single
token*, which is the whole reason it is the scale for playing:

- the confidence distribution is **one softmax row at one position** -- no
  candidate enumeration, no sequence scoring, no extra forward passes;
- the J-lens readout and the model's own distribution are measured at the
  **identical slot**, so a disagreement between them is a real disagreement and
  not an artefact of measuring two different things.

01b itself now runs on 0-100 for comparability with 2603.25052. There the lens
only ever sees the number's *first digit*, which is fine for a gate and no use
at all for understanding what the lens is doing.

The prompt is **prefilled up to `"Confidence: "`**, so position `-1` is the
confidence digit by construction -- no searching, no off-by-one.

In [10]:
SCALE = "nine"
CANDS = synthetic.confidence_candidates(scale=SCALE)     # 1..9
DIGIT_IDS = synthetic.digit_token_ids(tok)               # raises if multi-token
ITEMS = {i.item_id: i for i in synthetic.all_items(scale=SCALE)}

print("candidates:", CANDS)
print("digit token ids:", DIGIT_IDS)
print("items:", list(ITEMS))


@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 24) -> str:
    """Greedy continuation of an already-rendered prompt string."""
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


@torch.no_grad()
def slot_probs(prompt: str) -> np.ndarray:
    """Final-layer probabilities at the last position. On 1-9 this single row
    *is* the confidence distribution."""
    ids = tok(prompt, return_tensors="pt",
              add_special_tokens=False).input_ids.to(model.device)
    return torch.softmax(model(ids).logits[0, -1].float(), dim=-1).cpu().numpy()


def answer_of(item, max_new_tokens: int = 24) -> str:
    """First pass: the model's own answer, with any confidence line stripped."""
    rendered = tok.apply_chat_template(
        synthetic.build_chat(item), tokenize=False, add_generation_prompt=True
    )
    return generate(rendered, max_new_tokens).split("Confidence")[0].strip().split("\n")[0]


def confidence_prompt(item) -> str:
    """Full two-pass prompt, ending exactly at the confidence slot."""
    if isinstance(item, str):
        item = ITEMS[item]
    return synthetic.render(tok, item, answer_of(item))

candidates: [1, 2, 3, 4, 5, 6, 7, 8, 9]
digit token ids: {0: 236771, 1: 236770, 2: 236778, 3: 236800, 4: 236812, 5: 236810, 6: 236825, 7: 236832, 8: 236828, 9: 236819}
items: ['S1_002', 'S1_004', 'S1_006', 'S1_008', 'S2e_00', 'S2e_01', 'S2e_02', 'S2e_03', 'S2e_04', 'S2u_00', 'S2u_01', 'S2u_02', 'S2u_03', 'S2u_04']


In [11]:
def show_prompt(item, answer: str | None = "An answer.", n_tail_tokens: int = 12):
    """Print one item's full prompt and the tokens around the confidence slot.

    `answer` fixed  -> no GPU pass, inspect freely.
    `answer=None`   -> the real two-pass prompt, using the model's own answer.

    Prompt bugs are the cheapest way to manufacture a null and they are
    invisible in a summary statistic: a chat template that swallows the
    instruction and a lens that cannot see the digit look identical downstream.
    """
    if isinstance(item, str):
        item = ITEMS[item]
    ans = answer_of(item) if answer is None else answer
    prompt = synthetic.render(tok, item, ans)

    print(f"=== {item.item_id}  ({item.tier}, scale={item.scale}, target={item.target}) ===")
    print(f"--- answer used: {ans!r} {'(generated)' if answer is None else '(fixed)'}")
    print("--- rendered prompt, repr so the chat template is visible ---")
    print(repr(prompt))
    print("\n--- readable ---")
    print(prompt)

    ids = tok.encode(prompt, add_special_tokens=False)
    tail = ids[-n_tail_tokens:]
    print(f"--- last {len(tail)} tokens (position -1 is the confidence slot) ---")
    for offset, tid in zip(range(-len(tail), 0), tail):
        print(f"  {offset:>4}  {tid:>7}  {tok.decode([tid])!r}")

    assert not tok.decode([tail[-1]]).strip().isdigit(), (
        "prompt already ends on a digit -- the confidence slot is off by one")
    return prompt


# The two instructions on this scale, verbatim, before anything is interpreted.
print("S1 dictated:\n ", ITEMS["S1_004"].instruction)
print("\nS2 free:\n ", ITEMS["S2e_00"].instruction)

S1 dictated:
  Answer the question in one short sentence. Then on a new line write exactly `Confidence: 4`. Use that exact digit.

S2 free:
  Answer the question in one short sentence. Then on a new line write exactly `Confidence: N`, where N is a single digit from 1 (least confident) to 9 (most confident).


## 5. S1 - the dictated digit

The prompt names the digit to say, so the readout is copy-level -- the same
difficulty as Judy. That is the point: **if S1 fails, the readout is broken, not
the model.**

The prompt is printed first, in full, so the layer table below can be read
against the actual string that produced it. The digit the lens is being asked
to find is sitting in the instruction, in plain text, a few tokens back.

In [12]:
# The real two-pass prompt: `answer=None` runs the model's own first pass rather
# than the fixed placeholder, so this is the exact string the loop below feeds
# the lens. Greedy decoding, so it reproduces.
S1_PROMPT = show_prompt("S1_004", answer=None)

=== S1_004  (S1_dictated, scale=nine, target=4) ===
--- answer used: 'There are seven days in a week.' (generated)
--- rendered prompt, repr so the chat template is visible ---
'<bos><start_of_turn>user\nAnswer the question in one short sentence. Then on a new line write exactly `Confidence: 4`. Use that exact digit.\n\nQuestion: How many days are in a week?<end_of_turn>\n<start_of_turn>model\nThere are seven days in a week.\nConfidence: '

--- readable ---
<bos><start_of_turn>user
Answer the question in one short sentence. Then on a new line write exactly `Confidence: 4`. Use that exact digit.

Question: How many days are in a week?<end_of_turn>
<start_of_turn>model
There are seven days in a week.
Confidence: 
--- last 12 tokens (position -1 is the confidence slot) ---
   -12     3810  'There'
   -11      659  ' are'
   -10     6819  ' seven'
    -9     2668  ' days'
    -8      528  ' in'
    -7      496  ' a'
    -6     2069  ' week'
    -5   236761  '.'
    -4      107  '\n'
    -3

In [13]:
# That one prompt, all the way through the lens, before any aggregation.
s1_out = probe(S1_PROMPT, tokens="4")
print(f"\n'4' in the J-lens top-5 at layers: {hits(S1_PROMPT, '4', probs=s1_out)}")

'<bos><start_of_turn>user\nAnswer the question in one short sentence. Then on a new line write exactly `Confidence: 4`. Use that exact digit.\n\nQuestion: How many days are in a week?<end_of_turn>\n<start_of_turn>model\nThere are seven days in a week.\nConfidence: '

layer  J-lens                                                logit lens
    0  ['</strong>', '</h2>', '</h1>', ' t', '<start_of_image>']  ['^{\\', ':\\', ' percent', '$-', '^{-']
    1  [' t', ' ity', '<start_of_image>', ' h', ' i']        ['^{\\', ' ese', ':\\', ' percent', ' los']
    2  [' l', ' t', ' h', ' i', ' ih']                       ['^{\\', ' ese', ' los', ' esa', ':\\']
    3  [' l', ' mey', ' tb', ' jt', ' i']                    ['^{\\', ' ese', ' los', ':\\', ' esa']
    4  ['  ', '   ', ' l', ' i', ' t']                       [' that', ' ও', ' এই', '^{\\', ' and']
    5  ['  ', '<start_of_image>', ' l', ' i', ' Mos']        [' the', ' सभी', ' এই', ' that', ' এ']
    6  ['  ', ' ten', ' eleven', ' thirty', ' 


'4' in the J-lens top-5 at layers: [25, 26, 27, 28, 29, 30, 31, 32]


In [14]:
# Every dictated item, not just one. A criterion resting on a single prompt is a
# coin flip wearing a criterion's clothes.
s1_curves = {"J-lens": [], "logit lens": []}
s1_hits = {}

for item in synthetic.dictated_items(scale=SCALE):
    prompt = confidence_prompt(item)
    j_probs, l_probs, final = readout(prompt)
    wid = DIGIT_IDS[synthetic.digits_of(item.target)[0]]

    s1_curves["J-lens"].append([j_probs[l][wid] for l in LAYERS])
    s1_curves["logit lens"].append([l_probs[l][wid] for l in LAYERS])
    s1_hits[item.item_id] = hits(prompt, str(item.target),
                                 probs=(j_probs, l_probs, final))
    print(f"{item.item_id}  target={item.target}  P(final)={final[wid]:.3f}  "
          f"top-5 at layers {s1_hits[item.item_id] or 'never'}")

read_late = float(np.mean(
    [any(l >= N_LAYERS // 2 for l in ls) for ls in s1_hits.values()]))
print(f"\nitems whose digit is read in the top half of the stack: {read_late:.0%}")

viz.series_line(
    LAYERS, {k: np.mean(np.array(v), axis=0) for k, v in s1_curves.items()},
    y_range=(0, 1),
    title="S1 dictated: P(the dictated digit) at the confidence slot",
    xaxis="layer", yaxis="probability",
).show()

S1_002  target=2  P(final)=1.000  top-5 at layers [27, 28, 29, 30, 31, 32]
S1_004  target=4  P(final)=1.000  top-5 at layers [25, 26, 27, 28, 29, 30, 31, 32]
S1_006  target=6  P(final)=1.000  top-5 at layers [23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
S1_008  target=8  P(final)=1.000  top-5 at layers [23, 24, 25, 26, 27, 28, 29, 30, 31, 32]

items whose digit is read in the top half of the stack: 100%


In [15]:
# "Does a number go here" (robust) separately from "which number" (what we want).
# The first crystallizes earlier than the second, and conflating them is how a
# mid-stack digit-mass rise gets mistaken for the value being decided.
j_probs, l_probs, final = readout(S1_PROMPT)

grid = np.array([synthetic.digit_distribution(j_probs[l], DIGIT_IDS) for l in LAYERS])
viz.prob_heatmap(
    grid.T, x=LAYERS, y=list(range(10)),
    title="S1 (target 4): J-lens distribution over digits at the slot",
    xaxis="layer", yaxis="digit",
).show()

viz.series_line(
    LAYERS,
    {"J-lens": [synthetic.digit_mass(j_probs[l], DIGIT_IDS) for l in LAYERS],
     "logit lens": [synthetic.digit_mass(l_probs[l], DIGIT_IDS) for l in LAYERS]},
    y_range=(0, 1),
    title="Digit mass: where 'a number goes here' crystallizes",
    xaxis="layer", yaxis="P(token is a digit)",
).show()

## 6. S2 - the computed digit

The number is **never in the context**, so it has to be computed. Ground truth
is ordinal only: easy items should be rated above genuinely unanswerable ones.

The hard arm is unanswerable rather than merely difficult on purpose --
overconfidence on hard-but-answerable questions is the known phenomenon
(2604.01457) and would confound the direction being tested here.

This is the interesting half. S1 asks whether the lens can read a digit that is
already written down; S2 asks whether it can read one the model is in the middle
of deciding.

In [16]:
from collections import defaultdict

s2_dists, s2_expected, s2_prompts = defaultdict(list), defaultdict(list), {}

for item in synthetic.forced_extreme_items(scale=SCALE):
    prompt = confidence_prompt(item)
    s2_prompts[item.item_id] = prompt
    dist = synthetic.slot_distribution(slot_probs(prompt), DIGIT_IDS, CANDS)
    s2_dists[item.tier].append(dist)
    s2_expected[item.tier].append(synthetic.expected_confidence(CANDS, dist))
    print(f"{item.item_id}  {item.tier:<16} E[conf]="
          f"{s2_expected[item.tier][-1]:4.2f}/9   {item.question[:44]!r}")

easy = float(np.mean(s2_expected["S2_easy"]))
hard = float(np.mean(s2_expected["S2_unanswerable"]))
print(f"\nmean E[confidence]  easy={easy:.2f}  unanswerable={hard:.2f}  "
      f"separation={easy - hard:+.2f} points on a 1-9 scale")
if easy <= hard:
    print("   ^ the wrong direction. Worth understanding before anything else "
          "here is read as a lens result -- it is a fact about the model.")

viz.grouped_bar(
    CANDS,
    {"easy": np.mean(np.array(s2_dists["S2_easy"]), axis=0),
     "unanswerable": np.mean(np.array(s2_dists["S2_unanswerable"]), axis=0)},
    title="S2: confidence distribution at the slot (final layer)",
    xaxis="stated confidence (1-9)", yaxis="probability",
).show()

S2e_00  S2_easy          E[conf]=9.00/9   'What is 2 + 2?'
S2e_01  S2_easy          E[conf]=9.00/9   'What colour is the sky on a clear day?'
S2e_02  S2_easy          E[conf]=8.97/9   'How many legs does a spider have?'
S2e_03  S2_easy          E[conf]=9.00/9   'What is the capital of Japan?'
S2e_04  S2_easy          E[conf]=9.00/9   'Which is larger, an elephant or a mouse?'
S2u_00  S2_unanswerable  E[conf]=1.00/9   "What is my next-door neighbour's middle name"
S2u_01  S2_unanswerable  E[conf]=1.01/9   'How many coins are in my pocket right now?'
S2u_02  S2_unanswerable  E[conf]=1.00/9   'What did I eat for breakfast on 14 March 201'
S2u_03  S2_unanswerable  E[conf]=1.00/9   'What is the serial number of the laptop I am'
S2u_04  S2_unanswerable  E[conf]=1.00/9   'What time did my upstairs neighbour wake up '

mean E[confidence]  easy=8.99  unanswerable=1.00  separation=+7.99 points on a 1-9 scale


In [17]:
# The same slot, unaveraged. The tier means above can be bimodal even when no
# single prompt is, and `slot_distribution` renormalises over 1-9 -- so a tier
# with 10% of its mass on digits plots identically to one with 100%. Both are
# only visible per item.
s2_rows = {}
for item in synthetic.forced_extreme_items(scale=SCALE):
    p = slot_probs(s2_prompts[item.item_id])
    s2_rows[item.item_id] = (
        item.tier,
        synthetic.slot_distribution(p, DIGIT_IDS, CANDS),
        synthetic.digit_mass(p, DIGIT_IDS),      # before renormalisation
        top_k(p, 3),                             # what it would actually emit
    )

print(f"{'item':<9} {'tier':<16} {'says':>4} {'P|digit':>8} {'E[conf]':>8} "
      f"{'digit mass':>11}  top-3 tokens")
for iid, (tier, dist, dmass, tk) in s2_rows.items():
    k = int(np.argmax(dist))
    print(f"{iid:<9} {tier:<16} {CANDS[k]:>4} {dist[k]:>8.3f} "
          f"{synthetic.expected_confidence(CANDS, dist):>8.2f} {dmass:>11.3f}  "
          f"{[t for t, _ in tk]}")

order = sorted(s2_rows, key=lambda i: (s2_rows[i][0], i))
viz.prob_heatmap(
    np.array([s2_rows[i][1] for i in order]), x=CANDS, y=order,
    title="S2: confidence distribution per prompt (final layer, renormalised over 1-9)",
    xaxis="stated confidence (1-9)", yaxis="item",
).show()

viz.grouped_bar(
    order, {"P(token at the slot is a digit)": [s2_rows[i][2] for i in order]},
    title="S2: how much mass the renormalisation is throwing away",
    xaxis="item", yaxis="digit mass",
).show()


item      tier             says  P|digit  E[conf]  digit mass  top-3 tokens
S2e_00    S2_easy             9    1.000     9.00       1.000  ['9', '1', '8']
S2e_01    S2_easy             9    1.000     9.00       1.000  ['9', '8', '1']
S2e_02    S2_easy             9    0.997     8.97       1.000  ['9', '1', '8']
S2e_03    S2_easy             9    1.000     9.00       1.000  ['9', '1', '8']
S2e_04    S2_easy             9    1.000     9.00       1.000  ['9', '1', '8']
S2u_00    S2_unanswerable     1    0.999     1.00       1.000  ['1', '2', '3']
S2u_01    S2_unanswerable     1    0.991     1.01       1.000  ['1', '2', '3']
S2u_02    S2_unanswerable     1    0.997     1.00       1.000  ['1', '2', '3']
S2u_03    S2_unanswerable     1    0.998     1.00       1.000  ['1', '2', '3']
S2u_04    S2_unanswerable     1    0.998     1.00       1.000  ['1', '2', '3']


In [18]:
# Is there signal at all? Eyeballing two bars cannot answer this, and neither
# can a mean difference -- ground truth here is ordinal, so the statistic that
# matches it is a rank test on E[conf], easy vs unanswerable.
#
# Power note, read this before the p-value: 5 items per arm. The smallest p an
# exact one-sided rank-sum can return at 5v5 is 1/252 = 0.004, and that needs
# *perfect* separation. So this can confirm a strong effect and says nothing
# about a weak one -- a null here means "not enough items", not "no calibration".
from scipy.stats import mannwhitneyu

e_easy = [synthetic.expected_confidence(CANDS, d)
          for t, d, _, _ in s2_rows.values() if t == "S2_easy"]
e_hard = [synthetic.expected_confidence(CANDS, d)
          for t, d, _, _ in s2_rows.values() if t == "S2_unanswerable"]

u, p = mannwhitneyu(e_easy, e_hard, alternative="greater", method="exact")
auc = u / (len(e_easy) * len(e_hard))   # P(random easy item outranks random hard one)

print(f"model:            {cfg.name} ({cfg.n_params})")
print(f"easy         E[conf] = {np.mean(e_easy):.2f}  {np.round(e_easy, 2).tolist()}")
print(f"unanswerable E[conf] = {np.mean(e_hard):.2f}  {np.round(e_hard, 2).tolist()}")
print(f"separation           = {np.mean(e_easy) - np.mean(e_hard):+.2f} points on 1-9")
print(f"rank-sum (exact, one-sided)  AUC={auc:.2f}  p={p:.4f}")
print("  AUC 0.5 = no ordering; 1.0 = every easy item above every hard one")

# The floor this has to clear. If the digit distribution barely moves between
# arms there is nothing for the lens to read, and section 6 is measuring the
# lens against a constant.
spread = float(np.std([synthetic.expected_confidence(CANDS, d)
                       for _, d, _, _ in s2_rows.values()]))
print(f"\nspread of E[conf] across ALL items: sd={spread:.2f}")
if spread < 0.25:
    print("  ^ the model says essentially the same thing regardless of the "
          "question. That is a fact about the model, not about the J-lens, and "
          "no lens result below it can mean anything.")


model:            google/gemma-3-4b-it (4B)
easy         E[conf] = 8.99  [9.0, 9.0, 8.97, 9.0, 9.0]
unanswerable E[conf] = 1.00  [1.0, 1.01, 1.0, 1.0, 1.0]
separation           = +7.99 points on 1-9
rank-sum (exact, one-sided)  AUC=1.00  p=0.0040
  AUC 0.5 = no ordering; 1.0 = every easy item above every hard one

spread of E[conf] across ALL items: sd=4.00


In [19]:
# Does the computed digit appear in J-space before it is spoken? Track whichever
# digit the model actually commits to, per item.
s2_curves = {"J-lens": [], "logit lens": []}
for item in synthetic.forced_extreme_items(scale=SCALE):
    prompt = s2_prompts[item.item_id]
    j_probs, l_probs, final = readout(prompt)
    spoken = CANDS[int(np.argmax(synthetic.slot_distribution(final, DIGIT_IDS, CANDS)))]
    sid = DIGIT_IDS[spoken]
    s2_curves["J-lens"].append([j_probs[l][sid] for l in LAYERS])
    s2_curves["logit lens"].append([l_probs[l][sid] for l in LAYERS])

viz.series_line(
    LAYERS, {k: np.mean(np.array(v), axis=0) for k, v in s2_curves.items()},
    y_range=(0, 1),
    title="S2 computed: P(the digit the model goes on to say) at the slot",
    xaxis="layer", yaxis="probability",
).show()

In [20]:
# One unanswerable item end to end, prompt included -- the same treatment S1 got.
# Read the table against the prompt: nothing in the context names a number, so
# whatever the lens surfaces here was computed rather than copied.
S2_PROMPT = show_prompt("S2u_00", answer=None)
s2_out = probe(S2_PROMPT)

spoken = CANDS[int(np.argmax(
    synthetic.slot_distribution(s2_out[2], DIGIT_IDS, CANDS)))]
print(f"\nthe model settles on {spoken}; the J-lens has it in the top-5 at layers "
      f"{hits(S2_PROMPT, str(spoken), probs=s2_out) or 'never'}")

=== S2u_00  (S2_unanswerable, scale=nine, target=None) ===
--- answer used: 'I cannot answer that question as I have no information about your neighbors.' (generated)
--- rendered prompt, repr so the chat template is visible ---
"<bos><start_of_turn>user\nAnswer the question in one short sentence. Then on a new line write exactly `Confidence: N`, where N is a single digit from 1 (least confident) to 9 (most confident).\n\nQuestion: What is my next-door neighbour's middle name?<end_of_turn>\n<start_of_turn>model\nI cannot answer that question as I have no information about your neighbors.\nConfidence: "

--- readable ---
<bos><start_of_turn>user
Answer the question in one short sentence. Then on a new line write exactly `Confidence: N`, where N is a single digit from 1 (least confident) to 9 (most confident).

Question: What is my next-door neighbour's middle name?<end_of_turn>
<start_of_turn>model
I cannot answer that question as I have no information about your neighbors.
Confidence: 

## 7. Scratch

Yours. `probe(prompt)` for a layer table, `probe(prompt, " token")` to add the
per-layer curve, `hits(prompt, " token")` for just the layer list.

Things worth trying, roughly in order of how much they have told us before:

- **Paraphrase the instruction.** If the digit reads under one phrasing and not
  another, the readout is about the prompt and not the representation.
- **Drop the chat template.** Feed raw text ending in `"Confidence: "`. Judy
  reads as raw text, and it is worth knowing whether the template is what costs
  the digit.
- **Move the slot.** Ask for the confidence *before* the answer instead of
  after -- that is PLAN.md's P2, and the difference between them is the actual
  research question.
- **Sweep positions.** `readout(prompt, position=-2)` reads the token before the
  slot. Whether the digit is legible one token early says something about when
  it is committed to.

In [21]:
# Scratch. Nothing below this line is depended on by anything.
PROMPT = "The capital of France is"
_ = probe(PROMPT)

'The capital of France is'

layer  J-lens                                                logit lens
    0  ['</strong>', '</h1>', '</h2>', '<start_of_image>', '}\\']  ['否', 'entropic', 'ية', ' meant', ' throughput']
    1  ['<start_of_image>', '</strong>', '</h1>', '}.', '\\']  ['否', 'entropic', 'ية', ' الحال', ' throughput']
    2  ['<start_of_image>', '</h1>', '</strong>', '}.', '.}']  ['否', 'entropic', ' 물론', ' الحال', 'ojen']
    3  ['\\', '<start_of_image>', '</strong>', '</h1>', '.\\']  ['否', 'entropic', ' supposed', 'ojen', ' ales']
    4  ['  ', '</strong>', '</h1>', '   ', '<start_of_image>']  ['否', ' своего', ' 물론', '،', ' पूर्व']
    5  ['  ', '   ', '<start_of_image>', ' \\', ' \xad']     [' своего', ' 물론', 'và', '否', ' पूर्व']
    6  ['  ', '   ', ' â', 'Â', '�']                         [' 물론', ' своего', 'và', '否', '龴']
    7  ['  ', '<start_of_image>', ' \\', '   ', ' â']        [' 물론', 'và', ' своего', ' cities', ' debates']
    8  ['  ', ' Paris', '<start_of_image>', '